# Title

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
logging.basicConfig(level=logging.INFO)

In [43]:
import jax
import jax.numpy as np
from typing import Union, Optional
from random import randint
from numpy.random import randn

import torch
from torch import nn, Tensor

In [57]:
import tensorflow as tf

In [59]:
tf.RaggedTensor.from_nested_value_rowids?

Signature:
tf.RaggedTensor.from_nested_value_rowids(
    flat_values,
    nested_value_rowids,
    nested_nrows=None,
    name=None,
    validate=True,
)
Docstring:
Creates a `RaggedTensor` from a nested list of `value_rowids` tensors.

Equivalent to:

```python
result = flat_values
for (rowids, nrows) in reversed(zip(nested_value_rowids, nested_nrows)):
  result = from_value_rowids(result, rowids, nrows)
```

Args:
  flat_values: A potentially ragged tensor.
  nested_value_rowids: A list of 1-D integer tensors.  The `i`th tensor is
    used as the `value_rowids` for the `i`th ragged dimension.
  nested_nrows: A list of integer scalars.  The `i`th scalar is used as the
    `nrows` for the `i`th ragged dimension.
  name: A name prefix for the RaggedTensor (optional).
  validate: If true, then use assertions to check that the arguments form
    a valid `RaggedTensor`.  Note: these assertions incur a runtime cost,
      since they must be checked for each tensor value.

Returns:
  A `Ragg

In [52]:
class RecursiveNorm(nn.Module):
    def forward(self, x: Union[list, Tensor]) -> Tensor:
        if isinstance(x, list):
            return sum(self.forward(y) for y in x)
        return torch.linalg.norm(x)
    
module = RecursiveNorm()

In [54]:
def recursive_norm(x: Union[list[np.ndarray], np.ndarray]) -> np.ndarray:
    
    if isinstance(x, list):
        return sum(recursive_norm(y) for y in x)
    return np.linalg.norm(x)  

jitted_recursive_norm = jax.jit(recursive_norm)

In [56]:
jax.pmap(recursive_norm)(data)

ValueError: pmap got inconsistent sizes for array axes to be mapped:
the tree of axis sizes is:
([[1, 15, 1, 7]],)

In [46]:
max_length: int=16; max_rank: int=5
n = randint(1, max_rank+1)
shape = tuple(randint(1,max_length+1) for _ in range(n))
shape

(16, 3)

In [47]:

def random_tensor(max__length: int=16, max_rank: int=5):
    n = randint(1, max_rank)
    shape = tuple(randint(1,max_length) for _ in range(n))
    return np.array(randn(*shape))
    


def random_nested(max_length=5, max_depth=3, cur_depth=0):
    length = randint(1,5)
    if randint(0,1) and cur_depth<max_depth:
        # nest
        return [random_nested(cur_depth=cur_depth+1) for _ in range(length)]
    return [random_tensor() for _ in range(length)]

    
def to_torch(x, device: Optional[torch.device] = None):
    if isinstance(x, list):
        return [to_torch(y) for y in x]
    return torch.tensor(x.to_py())
    
    
tensor = random_tensor()
tensor.shape

(12, 13, 1)

In [48]:
data = random_nested()
torch_data = to_torch(data)

In [49]:
%%timeit
recursive_norm(data)

351 µs ± 27.6 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [51]:
%%timeit
jitted_recursive_norm(data)

25.7 µs ± 453 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [53]:
%%timeit
module(torch_data)

28.4 µs ± 649 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
